# Basic Multi-Agent Chat with Claude and AG2

This notebook demonstrates how to set up a basic two-agent conversation using [AG2](https://ag2.ai) (formerly AutoGen) with Claude as the underlying model.

AG2 is an open-source multi-agent framework with 500K+ monthly PyPI downloads. It provides high-level abstractions for building collaborative AI agent systems.

## What you'll learn
- How to configure AG2 to use Claude via Anthropic API
- How to create an AssistantAgent powered by Claude
- How to run a two-agent conversation

In [1]:
%pip install "ag2[anthropic]>=0.11.4,<1.0" -q

/Users/faridunm/Documents/WORK/AG2/Opensource/claude-cookbooks/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

from autogen import AssistantAgent, LLMConfig, UserProxyAgent

llm_config = LLMConfig(
    {
        "model": "claude-sonnet-4-6",
        "api_key": os.environ.get("ANTHROPIC_API_KEY"),
        "api_type": "anthropic",
    }
)

## Creating Agents

AG2 provides two key agent types:
- **AssistantAgent**: An AI-powered agent that generates responses using Claude
- **UserProxyAgent**: Represents the user or executes code on behalf of the user

In [3]:
assistant = AssistantAgent(
    name="Assistant",
    system_message=(
        "You are a helpful AI assistant. "
        "Provide clear, concise answers. "
        "Reply TERMINATE when the task is complete."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)

## Running the Conversation

We use `user_proxy.run(assistant, message=...)` to start a conversation. The agents will exchange messages until the termination condition is met.

In [4]:
chat = user_proxy.run(
    assistant,
    message="Explain the key differences between supervised and unsupervised learning in 3 bullet points.",
)
chat.process()

User (to Assistant):



Explain the key differences between supervised and unsupervised learning in 3 bullet points.



--------------------------------------------------------------------------------


Assistant (to User):



Here are the key differences between supervised and unsupervised learning:

• **Labeled Data** – Supervised learning uses **labeled training data** (input-output pairs), where the correct answers are provided during training. Unsupervised learning uses **unlabeled data**, requiring the algorithm to find patterns on its own without predefined outputs.

• **Goal & Task Type** – Supervised learning aims to **predict or classify** specific outcomes (e.g., spam detection, image recognition). Unsupervised learning aims to **discover hidden structure** or groupings within data (e.g., customer segmentation, anomaly detection).

• **Human Involvement & Complexity** – Supervised learning requires **more human effort** to label and curate data but produces more predictable, measurable results. Unsupervised learning demands **less upfront human input** but results can be harder to interpret and validate since there's no ground truth to compare against.

TERMINATE



--------------------------------------------------------------------------------



>>>>>>>> TERMINATING RUN (798b1cdf-2cfd-4f9d-8035-0661dcf09663): Termination message condition on agent 'User' met


/Users/faridunm/Documents/WORK/AG2/Opensource/claude-cookbooks/.venv/lib/python3.11/site-packages/autogen/oai/anthropic.py:1609: UserWarning: Cost calculation not available for model claude-sonnet-4-6
  warnings.warn(f"Cost calculation not available for model {model}", UserWarning)


## Accessing Results

The result object contains the full chat history, including all messages exchanged between agents.

In [5]:
for msg in chat.messages:
    name = msg.get("name", msg.get("role", "unknown"))
    content = msg.get("content", "")
    print(f"**{name}**: {content[:200]}")
    print("---")

**User**: Explain the key differences between supervised and unsupervised learning in 3 bullet points.
---
**Assistant**: Here are the key differences between supervised and unsupervised learning:

• **Labeled Data** – Supervised learning uses **labeled training data** (input-output pairs), where the correct answers are 
---


## Next Steps

- [Tool_Use_With_Agents.ipynb](./Tool_Use_With_Agents.ipynb) — Add function calling to your agents
- [GroupChat_Orchestration.ipynb](./GroupChat_Orchestration.ipynb) — Orchestrate multiple agents
- [AG2 Documentation](https://docs.ag2.ai) — Full framework reference